In [ ]:
!/usr/bin/env python3

In [4]:

import MDAnalysis as mda
from MDAnalysis.analysis import rms
from MDAnalysis.lib import distances 
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tkinter as tk
import subprocess
import readline
from pathlib import Path
import os

In [5]:
prmtop_file = '/data/bathir/EFE/EFE_WT/EFE-Mechanism/EFE-offline/WT/QMMM/Frame5039/Final/1-RC_Opt/rc.prmtop'
nc_file = '/data/bathir/EFE/EFE_WT/EFE-Mechanism/EFE-offline/WT/QMMM/Frame5039/Final/1-RC_Opt/rc.opt.pdb'
u = mda.Universe(prmtop_file, nc_file)
len(u.trajectory)
ag=u.atoms
print(ag.resids)

print(ag.resnames)
ag.select_atoms('not resname WAT').residues

[   1    1    1 ... 6142 6142 6142]
['MET' 'MET' 'MET' ... 'WAT' 'WAT' 'WAT']


<ResidueGroup with 346 residues>

In [17]:
# Function to calculate distances
def calculate_distances(resname1, name1, resname2, name2, prmtop_file, pdb_file, output_file, input_type, append=True):
    # Load the topology and PDB file
    u = mda.Universe(prmtop_file, pdb_file)

    # Select atoms
    selection1 = u.select_atoms(f'resname {resname1} and name {name1}')
    selection2 = u.select_atoms(f'resname {resname2} and name {name2}')
    
    # Check if selections are not empty
    if len(selection1) == 0 or len(selection2) == 0:
        print("One of the selections is empty. Please check your selection criteria.")
        return

    # Open output file for writing or appending based on the append parameter
    mode = 'a' if append else 'w'

    # Open output file for appending
    with open(output_file, mode) as f:
        f.write("-------------------------------------------------\n")
        
        # Calculate distances
        for atom1 in selection1:
            for atom2 in selection2:
                distance = np.linalg.norm(atom1.position - atom2.position)
                f.write(f"{atom1.resname}@{atom1.name}-{atom2.resname}@{atom2.name:<10}{distance:.2f}\n")
        
    print(f"Distances calculated and {'appended to' if append else 'written to'} {output_file}")
# Get the first input (RC or TS)
input_type = input("Enter the input type (RC or TS): ").strip().upper()

# Define file names based on input type
if input_type == "RC":
    prmtop_file = "rc.prmtop"
    pdb_file = "rc.opt.pdb"
elif input_type == "TS":
    prmtop_file = "ts.prmtop"
    pdb_file = "ts.opt.pdb"
elif input_type == "PD":
    prmtop_file = "pd.prmtop"
    pdb_file = "pd.opt.pdb"
else:
    print("Invalid input. Please enter 'RC' or 'TS' or 'PD'.")
    sys.exit(1)


# Example usage for multiple distance calculations
calculate_distances('AG1', 'C2', 'OY1', 'O2',prmtop_file,pdb_file,'Distance{input_type}.txt',input_type, append=False)
calculate_distances('AG1', 'C2', 'AG1', 'C1',prmtop_file,pdb_file,'Distance{input_type}.txt',input_type, append=True)
calculate_distances('AG1', 'C1', 'OY1', 'O1',prmtop_file,pdb_file,'Distance{input_type}.txt',input_type, append=True)
calculate_distances('AG1', 'C2', 'AG1', 'C3',prmtop_file,pdb_file,'Distance{input_type}.txt',input_type, append=True)
calculate_distances('AG1', 'C3', 'AG1', 'C4',prmtop_file,pdb_file,'Distance{input_type}.txt',input_type, append=True)
calculate_distances('AG1', 'C4', 'AG1', 'C5',prmtop_file,pdb_file,'Distance{input_type}.txt',input_type, append=True)
calculate_distances('AG1', 'C3', 'OY1', 'O1',prmtop_file,pdb_file,'Distance{input_type}.txt',input_type, append=True)


Distances calculated and appended to /data/bathir/EFE/EFE_WT/EFE-Mechanism/EFE-offline/WT/QMMM/Frame5039/Final/1-RC_Opt/Distance.txt
Distances calculated and appended to /data/bathir/EFE/EFE_WT/EFE-Mechanism/EFE-offline/WT/QMMM/Frame5039/Final/1-RC_Opt/Distance.txt
Distances calculated and appended to /data/bathir/EFE/EFE_WT/EFE-Mechanism/EFE-offline/WT/QMMM/Frame5039/Final/1-RC_Opt/Distance.txt
Distances calculated and appended to /data/bathir/EFE/EFE_WT/EFE-Mechanism/EFE-offline/WT/QMMM/Frame5039/Final/1-RC_Opt/Distance.txt
Distances calculated and appended to /data/bathir/EFE/EFE_WT/EFE-Mechanism/EFE-offline/WT/QMMM/Frame5039/Final/1-RC_Opt/Distance.txt
Distances calculated and appended to /data/bathir/EFE/EFE_WT/EFE-Mechanism/EFE-offline/WT/QMMM/Frame5039/Final/1-RC_Opt/Distance.txt
Distances calculated and appended to /data/bathir/EFE/EFE_WT/EFE-Mechanism/EFE-offline/WT/QMMM/Frame5039/Final/1-RC_Opt/Distance.txt


In [ ]:
# Load the trajectory and topology files
prmtop_file = '/data/bathir/EFE/EFE_WT/EFE-Mechanism/A198L/MD/EFE_A198L_solv.prmtop'
nc_file = '/data/bathir/EFE/EFE_WT/EFE-Mechanism/A198L/MD/Analysis/6-md_auto.nc'
u = mda.Universe(prmtop_file, nc_file)

# Initialize an array to store the counts of surrounding residues within 3.2 Å
counts_within_cutoff = []

# Iterate over each frame in the trajectory
for ts in u.trajectory:
    # Select surrounding residues within a certain distance cutoff (e.g., 3.2 Å)
    surrounding_residues = u.select_atoms(
        f'around 3.2 group non_standard_residue', non_standard_residue=non_standard_residue)

    # Count the number of surrounding residues within the cutoff distance
    count = len(surrounding_residues.residues)
    counts_within_cutoff.append(count)

# Convert counts to a numpy array for analysis
counts_within_cutoff = np.array(counts_within_cutoff)

# Plot histogram of the counts
plt.figure(figsize=(10, 6))
plt.hist(counts_within_cutoff, bins=np.arange(counts_within_cutoff.min(),
         counts_within_cutoff.max() + 1, 1), alpha=0.75, edgecolor='black')
plt.xlabel('Number of Surrounding Residues within 3.2 Å')
plt.ylabel('Frequency')
plt.title('Histogram of Number of Surrounding Residues within 3.2 Å')
plt.grid(True)
plt.tight_layout()

# Save and show the plot
plt.savefig('histogram_surrounding_residues.png', dpi=300)
plt.show()

In [ ]:
ag = u.select_atoms('resname AG1')
ag1 = u.select_atoms('resid 198')
da = distances.calc_bonds(ag.positions, ag1.positions, box=u.dimensions)
print('Our distance array has shape', da.shape)  
print(da)
ag.n_atoms
ag.radius_of_gyration()
view = nv.show_mdanalysis(ag)
view.camera = 'orthographic'
view

In [ ]:
# Calculate RMSD
rmsd = rms.RMSD(u, u, ref_frame=0, select='backbone')
rmsd.run()

# Convert frames to time
time_step = 0.1  # time step in nanoseconds
time = np.arange(len(rmsd.rmsd)) * time_step

# Calculate average RMSD
average_rmsd = np.mean(rmsd.rmsd[:, 2])

# Create a DataFrame with time and RMSD values
rmsd_df = pd.DataFrame({
    'Time (ns)': time,
    'RMSD (Å)': rmsd.rmsd[:, 2]
})

In [ ]:

# Print the DataFrame and average RMSD
print("RMSD DataFrame:")
print(rmsd_df)
print(f"\nAverage RMSD: {average_rmsd:.2f} Å")

In [ ]:
# Configure Matplotlib to use Arial font
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'

# Plot RMSD with high quality settings
plt.figure(figsize=(12, 8), dpi=300)  # Set figure size and resolution
plt.plot(time, rmsd.rmsd[:, 2], label='Backbone RMSD', linewidth=2)
plt.xlabel('Time (ns)', fontsize=24)
plt.ylabel('RMSD (Å)', fontsize=24)
plt.ylim(0, 10)
plt.xlim(-10,1010)
plt.title('RMSD', fontsize=32)
plt.legend(fontsize=24)
plt.grid(False)
plt.xticks(fontsize=24)
plt.yticks(fontsize=24)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate RMSF
rmsf = rms.RMSF(u.select_atoms('name CA')).run()
rmsf_values = rmsf.rmsf

In [ ]:
# Configure Matplotlib to use Arial font
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'

# Plot RMSF with high quality settings
plt.figure(figsize=(12, 8), dpi=300)  # Set figure size and resolution
plt.plot(rmsf.resids, rmsf_values, label='RMSF', linewidth=2)
plt.xlabel('Residue', fontsize=14)
plt.ylabel('RMSF (Å)', fontsize=14)
plt.title('RMSF', fontsize=16)
plt.legend(fontsize=12)
plt.ylim(0, 10)
plt.grid(False)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.savefig('rmsf_plot.png', dpi=300)  # Save the plot as a high-quality picture
plt.show()

In [ ]:

# Convert the notebook to a .py file
subprocess.run(["jupyter", "nbconvert", "--to",
               "script", "MD_Analysis.ipynb"])